<a href="https://colab.research.google.com/github/Lateephah/Applied-Search-Intelligence-System/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This notebook practices the same careful reading on my own week-5 model that this week's session
practiced on FlyRank's own research paper (*The State of AI-Driven SEO*, March 2026). The goal isn't to grade the paper, it holds itself to disclosed standards already, it's to bring that same rigor to my own work.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Lateephah/Applied-Search-Intelligence-System"
REPO_DIR = "Applied-Search-Intelligence-System"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

print("Working dir:", os.getcwd())

Working dir: /content/Applied-Search-Intelligence-System


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from?
Does the validation design support the claim?*

I picked both findings from the paper's own ML appendix, because that's the section closest to
what I'm doing in this lane, and the paper is admirably upfront that appendix pages are
"exploratory" and "secondary to direct aggregate comparisons", so these questions build on the
paper's own caution rather than contradicting it.

####**Finding A: "What Predicts Health?" (*Random Forest feature importance on Health Score, p.27*)**

The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top
three features a Random Forest leans on to predict Health Score — together with CTR, that's over
98% of the model's importance. The paper itself flags a caveat here: importance is "descriptive
rather than causal" because the target is "partly constructed from some of these inputs."

**My methodology question:** Health Score is explicitly defined elsewhere in the paper as a
weighted sum of Impressions (30 pts), Position (30 pts), CTR (20 pts), and Scroll Depth (20 pts).
If the target is a known linear formula of exactly four inputs, and the model's top four features
*are those same four inputs*, isn't the model mostly re-discovering the scoring formula's own
weights rather than learning anything new about content performance? This isn't a leakage
accusation in the classic sense (the paper discloses the overlap), but it does raise the question
of whether "feature importance" is the right frame at all here versus just decomposing a known
formula.

 **A constructive next step:** the same Random Forest predicting something genuinely
external to Health Score's definition like next-month impressions, or the observed growth/decline direction used elsewhere in the paper, would tell us whether these features predict *content performance* or just *recover arithmetic*.

####**Finding B: "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, p.29)**

The paper reports a Logistic Regression achieving 71% holdout accuracy separating growing from
declining pages, with an 80/20 split noted in the methodology section.

**My methodology question:** two things aren't stated that this week's validation checklist says to always check. First, **what's the base rate?** The paper doesn't report what share of the
holdout set was actually "growing" 71% accuracy on a label split close to 50/50 is a real
result, but 71% accuracy on a label that's already 65%+ one class is a much smaller edge than it
looks (the same "9 points of skill, not 71" trap the validation checklist calls out). Second,
**was the 80/20 split grouped by brand?** The dataset spans 57 brands with very different scales
(similar in spirit to my own 32-client dataset). If the split was a random row split rather than
grouped by brand, some of a brand's pages could sit in both train and holdout, which would let the
model partly learn brand-specific patterns rather than a signal that generalizes to a new brand,
exactly the gap I measure on my own model in section 2 below.

**Framed constructively:** reporting
the holdout base rate alongside the 71% figure, and confirming (or testing) whether the split was
brand-grouped, would let a reader judge how much of that number is real signal.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show the numbers.*

This directly tests my own methodology question from Finding B above, on my own data: what
happens to my week-5 Logistic Regression if I'd used a random row split instead of the
client-grouped split I actually used?

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

RAW_PATH = Path("data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(RAW_PATH)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Same honest feature set from w05 -- nothing derived from trend_direction/trend_pct/30d windows.
numeric_features = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "age_tier_order",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "search_volume", "competition", "cpc", "word_count", "char_count",
]
categorical_features = [
    "content_type", "main_intent", "competition_level", "freshness_tier",
    "word_count_tier", "char_count_tier", "impression_tier", "position_tier",
    "provider_used", "model_used",
]
feature_cols = numeric_features + categorical_features
X = df[feature_cols]
y = df["is_declining_label"]
groups = df["client_id"]

def make_pipe():
    preprocess = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ])
    return Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=2000, random_state=42))])

def precision_at_k(scores, labels, k):
    order = np.argsort(-scores)
    return np.asarray(labels)[order][:k].mean()

print(f"Rows: {len(df):,}   Clients: {groups.nunique()}")

Rows: 30,000   Clients: 32


In [3]:
# --- BEFORE: naive random row split (what the paper's methodology page does not rule out) ---
rs = ShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(rs.split(X, y))

pipe_random = make_pipe()
pipe_random.fit(X.iloc[tr_idx], y.iloc[tr_idx])
scores_random = pipe_random.predict_proba(X.iloc[te_idx])[:, 1]
y_test_random = y.iloc[te_idx].to_numpy()

overlap_random = len(set(df.iloc[tr_idx].client_id) & set(df.iloc[te_idx].client_id))
print(f"BEFORE -- random row split: {overlap_random} of {groups.nunique()} clients appear in BOTH train and test")
print(f"  ROC-AUC: {roc_auc_score(y_test_random, scores_random):.3f}   Avg Precision: {average_precision_score(y_test_random, scores_random):.3f}")
for k in [20, 50, 100]:
    print(f"  precision@{k}: {precision_at_k(scores_random, y_test_random, k):.3f}")

BEFORE -- random row split: 31 of 32 clients appear in BOTH train and test
  ROC-AUC: 0.704   Avg Precision: 0.718
  precision@20: 0.900
  precision@50: 0.880
  precision@100: 0.860


In [4]:
# --- AFTER: client-grouped split (what I actually used in w05) ---
gs = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx2, te_idx2 = next(gs.split(X, y, groups=groups))

pipe_grouped = make_pipe()
pipe_grouped.fit(X.iloc[tr_idx2], y.iloc[tr_idx2])
scores_grouped = pipe_grouped.predict_proba(X.iloc[te_idx2])[:, 1]
y_test_grouped = y.iloc[te_idx2].to_numpy()

overlap_grouped = len(set(df.iloc[tr_idx2].client_id) & set(df.iloc[te_idx2].client_id))
print(f"AFTER -- client-grouped split: {overlap_grouped} of {groups.nunique()} clients appear in BOTH train and test")
print(f"  ROC-AUC: {roc_auc_score(y_test_grouped, scores_grouped):.3f}   Avg Precision: {average_precision_score(y_test_grouped, scores_grouped):.3f}")
for k in [20, 50, 100]:
    print(f"  precision@{k}: {precision_at_k(scores_grouped, y_test_grouped, k):.3f}")

AFTER -- client-grouped split: 0 of 32 clients appear in BOTH train and test
  ROC-AUC: 0.577   Avg Precision: 0.567
  precision@20: 0.650
  precision@50: 0.640
  precision@100: 0.670


In [5]:
comparison = pd.DataFrame({
    "random split (BEFORE)": {
        "clients leaked across train/test": overlap_random,
        "ROC-AUC": round(roc_auc_score(y_test_random, scores_random), 3),
        "Avg Precision": round(average_precision_score(y_test_random, scores_random), 3),
        "precision@20": round(precision_at_k(scores_random, y_test_random, 20), 3),
        "precision@50": round(precision_at_k(scores_random, y_test_random, 50), 3),
        "precision@100": round(precision_at_k(scores_random, y_test_random, 100), 3),
    },
    "grouped split (AFTER)": {
        "clients leaked across train/test": overlap_grouped,
        "ROC-AUC": round(roc_auc_score(y_test_grouped, scores_grouped), 3),
        "Avg Precision": round(average_precision_score(y_test_grouped, scores_grouped), 3),
        "precision@20": round(precision_at_k(scores_grouped, y_test_grouped, 20), 3),
        "precision@50": round(precision_at_k(scores_grouped, y_test_grouped, 50), 3),
        "precision@100": round(precision_at_k(scores_grouped, y_test_grouped, 100), 3),
    },
}).T
comparison

,clients leaked across train/test,ROC-AUC,Avg Precision,precision@20,precision@50,precision@100
random split (BEFORE),31.0,0.704,0.718,0.90,0.88,0.86
grouped split (AFTER),0.0,0.577,0.567,0.65,0.64,0.67


**The gap, and why it exists:** the random split lets 31 of 32 clients appear in *both* train
and test, the model gets to see most of a client's other pages during training and is then
tested on that same client's held-out pages. The grouped split holds out entire clients (0 of 32
overlap). The random split's ROC-AUC (0.704) and precision@20 (0.900) look meaningfully better
than the grouped split's (0.577 and 0.650), **not because the model got smarter, but because it
partly memorized client-specific patterns** (a client's typical CTR, content style, or update
cadence) rather than learning something that transfers to a brand-new client.

This is the exact mechanism behind my methodology question on the paper's Finding B: if that
71%-accuracy Logistic Regression used a random split across 57 brands the way my "before" split
did here, its real generalization number to an *unseen* brand could plausibly be several points
lower, I can't know without the paper disclosing the split design, but I can now show concretely
what that kind of gap looks like on data shaped similarly to theirs.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Two checks: (1) confirm my test harness would actually catch a leak if one were present, by
deliberately adding known-leaky columns and watching the score jump; (2) confirm my real feature
set has none of them.

In [6]:
# Attack-your-own-model check: deliberately ADD leaky columns and watch the score jump.
# If it doesn't jump, the test harness itself is broken and nothing else in this audit is trustworthy.

def score_with_extra_features(extra_numeric):
    feats = numeric_features + extra_numeric
    Xe = df[feats + categorical_features]
    preprocess = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), feats),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ])
    pipe = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=2000, random_state=42))])
    gs2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    tr, te = next(gs2.split(Xe, y, groups=groups))
    pipe.fit(Xe.iloc[tr], y.iloc[tr])
    s = pipe.predict_proba(Xe.iloc[te])[:, 1]
    return roc_auc_score(y.iloc[te], s), average_precision_score(y.iloc[te], s)

clean_auc, clean_ap = roc_auc_score(y_test_grouped, scores_grouped), average_precision_score(y_test_grouped, scores_grouped)
leak1_auc, leak1_ap = score_with_extra_features(["trend_pct"])
leak2_auc, leak2_ap = score_with_extra_features(["impressions_last_30d", "impressions_prev_30d"])

print(f"Clean (honest) feature set              -> ROC-AUC {clean_auc:.3f}, AP {clean_ap:.3f}")
print(f"+ trend_pct (label-derived, Type 1)      -> ROC-AUC {leak1_auc:.3f}, AP {leak1_ap:.3f}")
print(f"+ last_30d/prev_30d windows (Type 2)     -> ROC-AUC {leak2_auc:.3f}, AP {leak2_ap:.3f}")

Clean (honest) feature set              -> ROC-AUC 0.577, AP 0.567
+ trend_pct (label-derived, Type 1)      -> ROC-AUC 0.989, AP 0.991
+ last_30d/prev_30d windows (Type 2)     -> ROC-AUC 0.836, AP 0.850


**The harness is sensitive, as it should be.** Adding `trend_pct` the literal column
`is_declining_label` is thresholded from  jumps ROC-AUC from 0.577 to 0.989. That's the
"collapse from ~1.0 to ~0.7" test from the leakage skill, run in reverse: going the other
direction confirms my clean pipeline isn't accidentally leaking, because it isn't anywhere near
1.0 already. Adding the raw last-30-day/prior-30-day windows the exact columns `trend_pct`
itself is built from, also jumps the score hard (0.836), confirming the *overlapping-window*
leak type independently of the *label-derived-column* leak type.

**Attack checklist, run against my real (clean) feature set:**
- [x] Timeline drawn: every feature is a trailing 90-day total or a point-in-time tier, never a
      last 30d/prev-30d window
- [x] No label-derived columns in the features: `trend_direction`, `trend_pct` excluded (confirmed
      above by testing WITH them and seeing the jump)
- [x] No product flags / existing-system scores as features (I use none of FlyRank's own
      Optimization Flags or Health Score as inputs, only raw GSC/GA4-style metrics)
- [x] Population selection checked: my filter (`impressions_90d > 0`, `content_age_days >= 90`) is
      a current-state visibility/age floor, not a fact about the outcome window
- [x] Split grouped by client (`client_id`), confirmed 0 overlap in section 2
- [x] Base rate printed next to every metric (0.542 population base rate; 0.511 test-slice base
      rate, both reported alongside precision@K in w05 and again above)
- [x] Top feature importance sanity-checked in w05: `days_with_impressions` and `content_age_days`
      lead, at ~0.02 importance each, nowhere near the ~1.0-importance signature a leaked column
      would show
- [x] Metrics recomputed out-of-fold: everything above is measured on the held-out test slice only
- [ ] Sealed/holdout claims: N/A, I haven't claimed a sealed holdout anywhere in this lane yet

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional,
decision-support.*

Auditing my own w05 notebook, two sentences go further than a single 80/20 split (one
`random_state`, no repeated folds, no confidence interval) can actually support.

**Original claim (w05, section 3):** *"my week-4 rule does not generalize to a new client"*

**What's wrong with it:** this is a flat, general statement built from exactly one train/test
split. A single grouped split tells me what happened on *this* particular 80/20 partition of 32
clients, it doesn't yet tell me the rule fails on client splits in general. The real, defensible
finding is narrower than the sentence I wrote.

**Rewrite:** "On this held-out group split, the week-4 rule's precision@20 dropped from 0.650
(measured on its own tuning population in week 4) to 0.500, at the test-slice base rate. That's
a directional signal that the rule may not generalize well to a client it wasn't built against,
but a single split isn't enough on its own to confirm it; repeated group splits (e.g. GroupKFold
across several held-out client sets) would be needed before treating this as a settled result."

---

**Original claim (w05, section 3):** *"Logistic Regression wins cleanly at every precision@K
cut"* ... *"I'd ship Logistic Regression"*

**What's wrong with it:** "wins cleanly" and "I'd ship" both state a production decision as
settled fact from one split with no variance estimate, I don't know how much these numbers
would move on a different random_state or a different 20% of clients.

**Rewrite:** "On this held-out group split, Logistic Regression's predicted probabilities produced
higher precision@20/50/100 than Random Forest's. That's decision-support evidence favoring
Logistic Regression for this lane, particularly because it's also the simpler, more readable
model  but it's measured on a single split, so I'd want to see the gap hold up across a few more
group splits before treating "Logistic Regression over Random Forest" as a settled choice rather
than a working recommendation."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.